In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [8]:
csv_path = "../data/raw/UPCT/CTD1_turbidity.csv"

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])
depth_cols = [c for c in df.columns if c.startswith('0.5')]
if not depth_cols:
    raise KeyError("No profundidad columns found")
def extract_depth(col):
    m = re.search(r'_(\-?\d+(\.\d+)?)', col)
    if not m:
        try:
            return float(col.split('_')[-1])
        except:
            return 0.0
    return float(m.group(1))
depth_cols = sorted(depth_cols, key=extract_depth, reverse=True)
df[depth_cols] = df[depth_cols].apply(pd.to_numeric, errors='coerce')
df = df[['Date'] + depth_cols]
df = df.drop_duplicates(subset=['Date'])
df = df.set_index('Date').sort_index()
df = df[~df.index.duplicated(keep='first')]
df = df.resample('D').mean()

missing_pct = df[depth_cols].isna().mean(axis=1)
df = df[missing_pct <= 0.5].copy()
if df.shape[0] < 2:
    raise ValueError("Not enough daily profiles after filtering")
df[depth_cols] = df[depth_cols].interpolate(axis=1, limit_direction='both')
df[depth_cols] = df[depth_cols].interpolate(axis=0, limit_direction='both')
df[depth_cols] = df[depth_cols].fillna(method='ffill').fillna(method='bfill')

X_raw = df[depth_cols].values.astype(float)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

n_components = min(2, X.shape[1])
pca = PCA(n_components=n_components, random_state=0)
X_pca = pca.fit_transform(X)

n_samples = X.shape[0]
max_k = min(10, max(2, n_samples - 1))
best_k = None
best_score = -1
scores = {}
for k in range(2, max_k + 1):
    km = KMeans(n_clusters=k, random_state=0, n_init=10)
    labels = km.fit_predict(X_pca)
    s = -1
    if len(set(labels)) > 1:
        try:
            s = silhouette_score(X_pca, labels)
        except:
            s = -1
    scores[k] = s
    if s > best_score:
        best_score = s
        best_k = k
if best_k is None:
    best_k = 2

km = KMeans(n_clusters=best_k, random_state=0, n_init=10)
labels = km.fit_predict(X_pca)
df['cluster'] = labels
df['missing_pct'] = missing_pct.reindex(df.index).fillna(0)

min_length = 7
groups = []
current_label = None
start = None
prev_idx = None
for idx, row in df.iterrows():
    lbl = int(row['cluster'])
    if current_label is None:
        current_label = lbl
        start = idx
    elif lbl != current_label:
        end = prev_idx
        length_days = (end - start).days + 1
        if length_days >= min_length:
            groups.append({'cluster': current_label, 'start': start, 'end': end, 'length_days': length_days})
        current_label = lbl
        start = idx
    prev_idx = idx
if current_label is not None and start is not None and prev_idx is not None:
    end = prev_idx
    length_days = (end - start).days + 1
    if length_days >= min_length:
        groups.append({'cluster': current_label, 'start': start, 'end': end, 'length_days': length_days})

out_dir = "../results/"
pd.DataFrame(groups).to_csv(out_dir + "analysis_candidate_groups.csv", index=False)
df.to_csv(out_dir + "analysis_profiles_with_clusters.csv")

plt.figure(figsize=(8,4))
ks = sorted(scores.keys())
plt.plot(ks, [scores[k] for k in ks], marker='o')
plt.xlabel('k')
plt.ylabel('silhouette score')
plt.title('Silhouette by k')
plt.tight_layout()
plt.savefig(out_dir + "silhouette_by_k.png")
plt.close()

plt.figure(figsize=(8,6))
palette = sns.color_palette("tab10", n_colors=max(2, best_k))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=labels, palette=palette, legend='full', s=40)
plt.title('PCA of daily profiles colored by cluster')
plt.tight_layout()
plt.savefig(out_dir + "pca_clusters.png")
plt.close()

plt.figure(figsize=(8,6))
for lbl in sorted(df['cluster'].unique()):
    mean_profile = df[df['cluster']==lbl][depth_cols].mean()
    plt.plot(range(len(mean_profile)), mean_profile.values, label=f'cluster {lbl}')
plt.legend()
plt.xlabel('depth index')
plt.ylabel('transparency')
plt.title('Mean profile per cluster')
plt.tight_layout()
plt.savefig(out_dir + "mean_profiles_per_cluster.png")
plt.close()

print("csv_path", str(csv_path))
print("best_k", best_k)
print("silhouette_scores", scores)
print("candidate groups saved to", str(out_dir + "analysis_candidate_groups.csv"))
print(pd.DataFrame(groups).to_string(index=False))

TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'

In [7]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

csv_path = "../data/IMIDA/CTD11_transparency.csv"

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]
if 'Date' not in df.columns:
    raise KeyError("Column 'Date' not found")
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', infer_datetime_format=True)
df = df.dropna(subset=['Date'])
depth_cols = [c for c in df.columns if c.startswith('0.5')]
if not depth_cols:
    raise KeyError("No profundidad columns found")
def extract_depth(col):
    m = re.search(r'_(\-?\d+(\.\d+)?)', col)
    if not m:
        try:
            return float(col.split('_')[-1])
        except:
            return 0.0
    return float(m.group(1))
depth_cols = sorted(depth_cols, key=extract_depth, reverse=True)
df[depth_cols] = df[depth_cols].apply(pd.to_numeric, errors='coerce')
df = df[['Date'] + depth_cols]
df = df.drop_duplicates(subset=['Date'])
df = df.set_index('Date').sort_index()
df = df[~df.index.duplicated(keep='first')]
df = df.resample('D').mean()

missing_pct = df[depth_cols].isna().mean(axis=1)
df = df[missing_pct <= 0.5].copy()
if df.shape[0] < 2:
    raise ValueError("Not enough daily profiles after filtering")
df[depth_cols] = df[depth_cols].interpolate(axis=1, limit_direction='both')
df[depth_cols] = df[depth_cols].interpolate(axis=0, limit_direction='both')
df[depth_cols] = df[depth_cols].fillna(method='ffill').fillna(method='bfill')

X_raw = df[depth_cols].values.astype(float)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

n_components = min(2, X.shape[1])
pca = PCA(n_components=n_components, random_state=0)
X_pca = pca.fit_transform(X)

n_samples = X.shape[0]
max_k = min(10, max(2, n_samples - 1))
best_k = None
best_score = -1
scores = {}
for k in range(2, max_k + 1):
    km = KMeans(n_clusters=k, random_state=0, n_init=10)
    labels = km.fit_predict(X_pca)
    s = -1
    if len(set(labels)) > 1:
        try:
            s = silhouette_score(X_pca, labels)
        except:
            s = -1
    scores[k] = s
    if s > best_score:
        best_score = s
        best_k = k
if best_k is None:
    best_k = 2

km = KMeans(n_clusters=best_k, random_state=0, n_init=10)
labels = km.fit_predict(X_pca)
df['cluster'] = labels
df['missing_pct'] = missing_pct.reindex(df.index).fillna(0)

min_length = 7
groups = []
current_label = None
start = None
prev_idx = None
for idx, row in df.iterrows():
    lbl = int(row['cluster'])
    if current_label is None:
        current_label = lbl
        start = idx
    elif lbl != current_label:
        end = prev_idx
        length_days = (end - start).days + 1
        if length_days >= min_length:
            groups.append({'cluster': current_label, 'start': start, 'end': end, 'length_days': length_days})
        current_label = lbl
        start = idx
    prev_idx = idx
if current_label is not None and start is not None and prev_idx is not None:
    end = prev_idx
    length_days = (end - start).days + 1
    if length_days >= min_length:
        groups.append({'cluster': current_label, 'start': start, 'end': end, 'length_days': length_days})

out_dir = "../results/"
pd.DataFrame(groups).to_csv(out_dir + "analysis_candidate_groups.csv", index=False)
df.to_csv(out_dir + "analysis_profiles_with_clusters.csv")

plt.figure(figsize=(8,4))
ks = sorted(scores.keys())
plt.plot(ks, [scores[k] for k in ks], marker='o')
plt.xlabel('k')
plt.ylabel('silhouette score')
plt.title('Silhouette by k')
plt.tight_layout()
plt.savefig(out_dir + "silhouette_by_k.png")
plt.close()

plt.figure(figsize=(8,6))
palette = sns.color_palette("tab10", n_colors=max(2, best_k))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=labels, palette=palette, legend='full', s=40)
plt.title('PCA of daily profiles colored by cluster')
plt.tight_layout()
plt.savefig(out_dir + "pca_clusters.png")
plt.close()

plt.figure(figsize=(8,6))
for lbl in sorted(df['cluster'].unique()):
    mean_profile = df[df['cluster']==lbl][depth_cols].mean()
    plt.plot(range(len(mean_profile)), mean_profile.values, label=f'cluster {lbl}')
plt.legend()
plt.xlabel('depth index')
plt.ylabel('transparency')
plt.title('Mean profile per cluster')
plt.tight_layout()
plt.savefig(out_dir + "mean_profiles_per_cluster.png")
plt.close()

print("csv_path", str(csv_path))
print("best_k", best_k)
print("silhouette_scores", scores)
print("candidate groups saved to", str(out_dir + "analysis_candidate_groups.csv"))
print(pd.DataFrame(groups).to_string(index=False))

FileNotFoundError: [Errno 2] No such file or directory: '../data/IMIDA/CTD11_transparency.csv'

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

csv_path = "../data/IMIDA/CTD2_turbidity.csv"

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]
if 'Date' not in df.columns:
    raise KeyError("Column 'Date' not found")
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', infer_datetime_format=True)
df = df.dropna(subset=['Date'])
depth_cols = [c for c in df.columns if c.startswith('profundidad')]
if not depth_cols:
    raise KeyError("No profundidad columns found")
def extract_depth(col):
    m = re.search(r'_(\-?\d+(\.\d+)?)', col)
    if not m:
        try:
            return float(col.split('_')[-1])
        except:
            return 0.0
    return float(m.group(1))
depth_cols = sorted(depth_cols, key=extract_depth, reverse=True)
df[depth_cols] = df[depth_cols].apply(pd.to_numeric, errors='coerce')
df = df[['Date'] + depth_cols]
df = df.drop_duplicates(subset=['Date'])
df = df.set_index('Date').sort_index()
df = df[~df.index.duplicated(keep='first')]
df = df.resample('D').mean()

missing_pct = df[depth_cols].isna().mean(axis=1)
df = df[missing_pct <= 0.5].copy()
if df.shape[0] < 2:
    raise ValueError("Not enough daily profiles after filtering")
df[depth_cols] = df[depth_cols].interpolate(axis=1, limit_direction='both')
df[depth_cols] = df[depth_cols].interpolate(axis=0, limit_direction='both')
df[depth_cols] = df[depth_cols].fillna(method='ffill').fillna(method='bfill')

X_raw = df[depth_cols].values.astype(float)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

n_components = min(2, X.shape[1])
pca = PCA(n_components=n_components, random_state=0)
X_pca = pca.fit_transform(X)

n_samples = X.shape[0]
max_k = min(10, max(2, n_samples - 1))
best_k = None
best_score = -1
scores = {}
for k in range(2, max_k + 1):
    km = KMeans(n_clusters=k, random_state=0, n_init=10)
    labels = km.fit_predict(X_pca)
    s = -1
    if len(set(labels)) > 1:
        try:
            s = silhouette_score(X_pca, labels)
        except:
            s = -1
    scores[k] = s
    if s > best_score:
        best_score = s
        best_k = k
if best_k is None:
    best_k = 2

km = KMeans(n_clusters=best_k, random_state=0, n_init=10)
labels = km.fit_predict(X_pca)
df['cluster'] = labels
df['missing_pct'] = missing_pct.reindex(df.index).fillna(0)

min_length = 7
groups = []
current_label = None
start = None
prev_idx = None
for idx, row in df.iterrows():
    lbl = int(row['cluster'])
    if current_label is None:
        current_label = lbl
        start = idx
    elif lbl != current_label:
        end = prev_idx
        length_days = (end - start).days + 1
        if length_days >= min_length:
            groups.append({'cluster': current_label, 'start': start, 'end': end, 'length_days': length_days})
        current_label = lbl
        start = idx
    prev_idx = idx
if current_label is not None and start is not None and prev_idx is not None:
    end = prev_idx
    length_days = (end - start).days + 1
    if length_days >= min_length:
        groups.append({'cluster': current_label, 'start': start, 'end': end, 'length_days': length_days})

out_dir = "../results/"
pd.DataFrame(groups).to_csv(out_dir + "analysis_candidate_groups.csv", index=False)
df.to_csv(out_dir + "analysis_profiles_with_clusters.csv")

plt.figure(figsize=(8,4))
ks = sorted(scores.keys())
plt.plot(ks, [scores[k] for k in ks], marker='o')
plt.xlabel('k')
plt.ylabel('silhouette score')
plt.title('Silhouette by k')
plt.tight_layout()
plt.savefig(out_dir + "silhouette_by_k.png")
plt.close()

plt.figure(figsize=(8,6))
palette = sns.color_palette("tab10", n_colors=max(2, best_k))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=labels, palette=palette, legend='full', s=40)
plt.title('PCA of daily profiles colored by cluster')
plt.tight_layout()
plt.savefig(out_dir + "pca_clusters.png")
plt.close()

plt.figure(figsize=(8,6))
for lbl in sorted(df['cluster'].unique()):
    mean_profile = df[df['cluster']==lbl][depth_cols].mean()
    plt.plot(range(len(mean_profile)), mean_profile.values, label=f'cluster {lbl}')
plt.legend()
plt.xlabel('depth index')
plt.ylabel('transparency')
plt.title('Mean profile per cluster')
plt.tight_layout()
plt.savefig(out_dir + "mean_profiles_per_cluster.png")
plt.close()

print("csv_path", str(csv_path))
print("best_k", best_k)
print("silhouette_scores", scores)
print("candidate groups saved to", str(out_dir + "analysis_candidate_groups.csv"))
print(pd.DataFrame(groups).to_string(index=False))

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

csv_path = "../data/IMIDA/CTD4_turbidity.csv"

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]
if 'Date' not in df.columns:
    raise KeyError("Column 'Date' not found")
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', infer_datetime_format=True)
df = df.dropna(subset=['Date'])
depth_cols = [c for c in df.columns if c.startswith('profundidad')]
if not depth_cols:
    raise KeyError("No profundidad columns found")
def extract_depth(col):
    m = re.search(r'_(\-?\d+(\.\d+)?)', col)
    if not m:
        try:
            return float(col.split('_')[-1])
        except:
            return 0.0
    return float(m.group(1))
depth_cols = sorted(depth_cols, key=extract_depth, reverse=True)
df[depth_cols] = df[depth_cols].apply(pd.to_numeric, errors='coerce')
df = df[['Date'] + depth_cols]
df = df.drop_duplicates(subset=['Date'])
df = df.set_index('Date').sort_index()
df = df[~df.index.duplicated(keep='first')]
df = df.resample('D').mean()

missing_pct = df[depth_cols].isna().mean(axis=1)
df = df[missing_pct <= 0.5].copy()
if df.shape[0] < 2:
    raise ValueError("Not enough daily profiles after filtering")
df[depth_cols] = df[depth_cols].interpolate(axis=1, limit_direction='both')
df[depth_cols] = df[depth_cols].interpolate(axis=0, limit_direction='both')
df[depth_cols] = df[depth_cols].fillna(method='ffill').fillna(method='bfill')

X_raw = df[depth_cols].values.astype(float)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

n_components = min(2, X.shape[1])
pca = PCA(n_components=n_components, random_state=0)
X_pca = pca.fit_transform(X)

n_samples = X.shape[0]
max_k = min(10, max(2, n_samples - 1))
best_k = None
best_score = -1
scores = {}
for k in range(2, max_k + 1):
    km = KMeans(n_clusters=k, random_state=0, n_init=10)
    labels = km.fit_predict(X_pca)
    s = -1
    if len(set(labels)) > 1:
        try:
            s = silhouette_score(X_pca, labels)
        except:
            s = -1
    scores[k] = s
    if s > best_score:
        best_score = s
        best_k = k
if best_k is None:
    best_k = 2

km = KMeans(n_clusters=best_k, random_state=0, n_init=10)
labels = km.fit_predict(X_pca)
df['cluster'] = labels
df['missing_pct'] = missing_pct.reindex(df.index).fillna(0)

min_length = 7
groups = []
current_label = None
start = None
prev_idx = None
for idx, row in df.iterrows():
    lbl = int(row['cluster'])
    if current_label is None:
        current_label = lbl
        start = idx
    elif lbl != current_label:
        end = prev_idx
        length_days = (end - start).days + 1
        if length_days >= min_length:
            groups.append({'cluster': current_label, 'start': start, 'end': end, 'length_days': length_days})
        current_label = lbl
        start = idx
    prev_idx = idx
if current_label is not None and start is not None and prev_idx is not None:
    end = prev_idx
    length_days = (end - start).days + 1
    if length_days >= min_length:
        groups.append({'cluster': current_label, 'start': start, 'end': end, 'length_days': length_days})

out_dir = "../results/"
pd.DataFrame(groups).to_csv(out_dir + "analysis_candidate_groups.csv", index=False)
df.to_csv(out_dir + "analysis_profiles_with_clusters.csv")

plt.figure(figsize=(8,4))
ks = sorted(scores.keys())
plt.plot(ks, [scores[k] for k in ks], marker='o')
plt.xlabel('k')
plt.ylabel('silhouette score')
plt.title('Silhouette by k')
plt.tight_layout()
plt.savefig(out_dir + "silhouette_by_k.png")
plt.close()

plt.figure(figsize=(8,6))
palette = sns.color_palette("tab10", n_colors=max(2, best_k))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=labels, palette=palette, legend='full', s=40)
plt.title('PCA of daily profiles colored by cluster')
plt.tight_layout()
plt.savefig(out_dir + "pca_clusters.png")
plt.close()

plt.figure(figsize=(8,6))
for lbl in sorted(df['cluster'].unique()):
    mean_profile = df[df['cluster']==lbl][depth_cols].mean()
    plt.plot(range(len(mean_profile)), mean_profile.values, label=f'cluster {lbl}')
plt.legend()
plt.xlabel('depth index')
plt.ylabel('transparency')
plt.title('Mean profile per cluster')
plt.tight_layout()
plt.savefig(out_dir + "mean_profiles_per_cluster.png")
plt.close()

print("csv_path", str(csv_path))
print("best_k", best_k)
print("silhouette_scores", scores)
print("candidate groups saved to", str(out_dir + "analysis_candidate_groups.csv"))
print(pd.DataFrame(groups).to_string(index=False))